<a href="https://colab.research.google.com/github/sarahliu-mma/Quantitative-Trading-Pipeline_FIAM-Hackathon-2026/blob/main/notebooks/02_ridge_model_prediction_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt

In [17]:
from sklearn.metrics import mean_squared_error

In [3]:
mt = pd.read_parquet('/content/drive/MyDrive/FIAM Hackathon/model_table.parquet')
mt.head()

,permno,eom,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,at_turnover,...,taccruals_ni,tangibility,tax_gr1a,turnover_126d,turnover_var_126d,z_score,zero_trades_126d,zero_trades_21d,zero_trades_252d,ret_exc_lead1m
0,10001,2015-01-31,0.689607,0.319399,0.819856,0.814019,0.407899,0.483486,0.726124,0.515982,...,0.301721,0.391886,0.177179,0.339908,0.782318,0.211921,0.658683,0.682250,0.510819,0.035918
1,10025,2015-01-31,0.712254,0.433847,0.703504,0.805565,0.802513,0.118412,0.697683,0.961012,...,0.093783,0.976579,0.075833,0.292356,0.690032,0.576987,0.706235,0.624956,0.592409,0.010997
2,10026,2015-01-31,0.712254,0.422890,0.276730,0.466009,0.059964,0.507027,0.176615,0.813839,...,0.620302,0.681305,0.627215,0.201127,0.565692,0.915149,0.796759,0.682601,0.786449,0.031309
4,10032,2015-01-31,0.689607,0.755276,0.822552,0.480099,0.433034,0.490513,0.586025,0.873902,...,0.876712,0.771225,0.587172,0.404368,0.114477,0.611755,0.594576,0.494903,0.610500,0.062307
5,10044,2015-01-31,0.712254,0.808036,0.382300,0.901726,0.374506,0.906535,0.224017,0.806814,...,0.232174,0.565872,0.350461,0.116590,0.786897,0.737997,0.879183,0.771529,0.903157,0.129391


In [4]:
IDENTIFIER = ['permno','eom']
TARGET = 'ret_exc_lead1m'

In [5]:
FEATURES = [f for f in mt.columns.tolist() if f not in IDENTIFIER+[TARGET]]

In [6]:
mt.info()

<class 'pandas.core.frame.DataFrame'>
Index: 376013 entries, 0 to 529081
Columns: 150 entries, permno to ret_exc_lead1m
dtypes: datetime64[ns](1), float64(148), int64(1)
memory usage: 433.2 MB


In [7]:
mt[FEATURES].select_dtypes('object')

""
0
1
2
4
5
...
529077
529078
529079
529080


In [8]:
mt = mt.drop(columns=mt[FEATURES].select_dtypes('object').columns.tolist())

In [9]:
FEATURES = [f for f in mt.columns.tolist() if f not in IDENTIFIER+[TARGET]]

In [10]:
mt = mt.sort_values(['eom','permno']).reset_index(drop= True)

In [11]:
def data_split(dt, features, train_start, train_end, val_window=1):
  year = dt['eom'].dt.year
  train = dt[(year >= train_start) & (year <= train_end)].dropna(subset=[TARGET])
  val   = dt[(year > train_end) & (year <= train_end + val_window)].dropna(subset=[TARGET])
  return (train[features].fillna(0.5), val[features].fillna(0.5),
            train[TARGET], val[TARGET], val[['permno', 'eom']])

In [12]:
def best_alpha(dt,alpha_list,features, train_start, train_end, val_window):
  alpha = np.array(alpha_list)
  train_x, val_x, train_y, val_y, val_ids = data_split(dt,features, train_start,train_end, val_window)
  score = {}
  for a in alpha:
    model = Ridge(alpha = a).fit(train_x,train_y)
    val_pred = model.predict(val_x)
    val_result = val_ids.copy()
    val_result['pred'] = val_pred
    val_result['y'] = val_y
    monthly_ic = val_result.groupby('eom').apply(lambda g: g['pred'].corr(g['y'], method = 'spearman'))
    score[a] = monthly_ic.mean()

  best = max(score, key = score.get)
  return score, best


In [13]:
alpha = [10,100,1000,10000,100000]
train_window = 3

for train_end in range(2017, 2020):
    train_start = train_end - train_window + 1
    score, best = best_alpha(mt, alpha, FEATURES, train_start, train_end, 1)
    print(f"Train {train_start}-{train_end}, validate {train_end + 1}: best alpha = {best}")
    print(score)
    print('-' * 30)


/tmp/ipykernel_2542/565111808.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly_ic = val_result.groupby('eom').apply(lambda g: g['pred'].corr(g['y'], method = 'spearman'))
/tmp/ipykernel_2542/565111808.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly_ic = val_result.groupby('eom').apply(lambda g: g['pred'].corr(g['y'], method = 'spearman'))
/tmp/ipykernel_2542/565111808.py:11: Depreca

Train 2015-2017, validate 2018: best alpha = 10000
{np.int64(10): np.float64(0.03663527898369387), np.int64(100): np.float64(0.0380232475681543), np.int64(1000): np.float64(0.03869435852070072), np.int64(10000): np.float64(0.039500397757749356), np.int64(100000): np.float64(0.0368051447713035)}
------------------------------


/tmp/ipykernel_2542/565111808.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly_ic = val_result.groupby('eom').apply(lambda g: g['pred'].corr(g['y'], method = 'spearman'))
/tmp/ipykernel_2542/565111808.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly_ic = val_result.groupby('eom').apply(lambda g: g['pred'].corr(g['y'], method = 'spearman'))
/tmp/ipykernel_2542/565111808.py:11: Depreca

Train 2016-2018, validate 2019: best alpha = 1000
{np.int64(10): np.float64(0.03687776990384369), np.int64(100): np.float64(0.04012459889644395), np.int64(1000): np.float64(0.0458275706052799), np.int64(10000): np.float64(0.03714028188323557), np.int64(100000): np.float64(0.018117882447701582)}
------------------------------


/tmp/ipykernel_2542/565111808.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly_ic = val_result.groupby('eom').apply(lambda g: g['pred'].corr(g['y'], method = 'spearman'))
/tmp/ipykernel_2542/565111808.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly_ic = val_result.groupby('eom').apply(lambda g: g['pred'].corr(g['y'], method = 'spearman'))
/tmp/ipykernel_2542/565111808.py:11: Depreca

Train 2017-2019, validate 2020: best alpha = 100000
{np.int64(10): np.float64(0.009046320325703181), np.int64(100): np.float64(0.013571376834862462), np.int64(1000): np.float64(0.01912923598211513), np.int64(10000): np.float64(0.02608148749078969), np.int64(100000): np.float64(0.031621905787154105)}
------------------------------


/tmp/ipykernel_2542/565111808.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly_ic = val_result.groupby('eom').apply(lambda g: g['pred'].corr(g['y'], method = 'spearman'))


In [14]:
final_alpha = 10000

def ridge_predict(dt, alpha, pred_year, rolling_window):
    year = dt['eom'].dt.year
    train_roll_start = pred_year - rolling_window
    train = dt[(year >= train_roll_start) & (year < pred_year) ].dropna(subset=[TARGET])
    test = dt[year == pred_year]
    model = Ridge(alpha = alpha).fit(train[FEATURES].fillna(0.5),train[TARGET])
    test_pred = model.predict(test[FEATURES].fillna(0.5))
    result = test.copy()
    result['pred'] = test_pred
    result['y'] = dt[TARGET]
    monthly_ic = result.groupby('eom').apply(lambda g: g['pred'].corr(g['y'], method = 'spearman'))
    score = monthly_ic.mean()
    return(result, monthly_ic, score)

In [19]:
for y in range(2021,2027):
  result, monthly_ic, score = ridge_predict(mt, final_alpha, y, 3)
  valid = result.dropna(subset=[TARGET])
  rmse = valid.groupby(valid['eom'].dt.year).apply(lambda g: np.sqrt(mean_squared_error(g[TARGET], g['pred'])))
  r2 = 1 - ((valid[TARGET] - valid['pred'])**2).sum() / (valid[TARGET]**2).sum()
  print('Year',y)
  print('Monthly IC in this year:',monthly_ic)
  print('Average monthly IC:',score)
  print('RMSE:',rmse)
  print('R2:',r2)

/tmp/ipykernel_2542/3920933891.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly_ic = result.groupby('eom').apply(lambda g: g['pred'].corr(g['y'], method = 'spearman'))


Year 2021
Monthly IC in this year: eom
2021-01-31   -0.015308
2021-02-28   -0.390669
2021-03-31   -0.051125
2021-04-30   -0.210714
2021-05-31    0.283093
2021-06-30   -0.319318
2021-07-31   -0.028174
2021-08-31   -0.195476
2021-09-30   -0.035764
2021-10-31   -0.252300
2021-11-30   -0.294073
2021-12-31   -0.507067
dtype: float64
Average monthly IC: -0.16807465743274916
RMSE: eom
2021    0.139148
dtype: float64
R2: -0.019497786329972122


/tmp/ipykernel_2542/3920933891.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly_ic = result.groupby('eom').apply(lambda g: g['pred'].corr(g['y'], method = 'spearman'))


Year 2022
Monthly IC in this year: eom
2022-01-31    0.104916
2022-02-28   -0.014841
2022-03-31    0.302966
2022-04-30    0.136069
2022-05-31   -0.160984
2022-06-30    0.064711
2022-07-31   -0.022252
2022-08-31   -0.026912
2022-09-30    0.195763
2022-10-31    0.030674
2022-11-30   -0.072436
2022-12-31    0.088688
dtype: float64
Average monthly IC: 0.0521968036099269
RMSE: eom
2022    0.153026
dtype: float64
R2: -0.007703019617683049


/tmp/ipykernel_2542/3920933891.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly_ic = result.groupby('eom').apply(lambda g: g['pred'].corr(g['y'], method = 'spearman'))


Year 2023
Monthly IC in this year: eom
2023-01-31    0.015543
2023-02-28    0.009460
2023-03-31    0.022497
2023-04-30   -0.023528
2023-05-31    0.244799
2023-06-30   -0.033395
2023-07-31    0.101390
2023-08-31    0.093161
2023-09-30   -0.057098
2023-10-31    0.014299
2023-11-30   -0.148720
2023-12-31    0.102497
dtype: float64
Average monthly IC: 0.02840877577035308
RMSE: eom
2023    0.144664
dtype: float64
R2: -0.006055894936065176


/tmp/ipykernel_2542/3920933891.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly_ic = result.groupby('eom').apply(lambda g: g['pred'].corr(g['y'], method = 'spearman'))


Year 2024
Monthly IC in this year: eom
2024-01-31   -0.128832
2024-02-29    0.226722
2024-03-31    0.121675
2024-04-30    0.090375
2024-05-31    0.035052
2024-06-30    0.124448
2024-07-31    0.020554
2024-08-31    0.056812
2024-09-30   -0.059649
2024-10-31    0.080085
2024-11-30    0.052335
2024-12-31    0.032476
dtype: float64
Average monthly IC: 0.05433777417260033
RMSE: eom
2024    0.149261
dtype: float64
R2: -0.0018412637049054492


/tmp/ipykernel_2542/3920933891.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly_ic = result.groupby('eom').apply(lambda g: g['pred'].corr(g['y'], method = 'spearman'))


Year 2025
Monthly IC in this year: eom
2025-01-31    0.109033
2025-02-28    0.005256
2025-03-31    0.001062
2025-04-30    0.096118
2025-05-31    0.034317
2025-06-30   -0.051040
2025-07-31    0.064091
2025-08-31   -0.023503
2025-09-30   -0.090392
2025-10-31    0.105240
2025-11-30    0.110013
2025-12-31    0.024416
dtype: float64
Average monthly IC: 0.032050933015924384
RMSE: eom
2025    0.151224
dtype: float64
R2: -0.0005668113247314643
Year 2026
Monthly IC in this year: eom
2026-01-31    0.000175
2026-02-28    0.078229
2026-03-31    0.133491
2026-04-30    0.068985
2026-05-31    0.049116
2026-06-30    0.004514
2026-07-31   -0.041521
2026-08-31         NaN
dtype: float64
Average monthly IC: 0.04185559935670128
RMSE: eom
2026    0.169924
dtype: float64
R2: 0.006860328758676926


/tmp/ipykernel_2542/3920933891.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  monthly_ic = result.groupby('eom').apply(lambda g: g['pred'].corr(g['y'], method = 'spearman'))
